# Phase 1 EDA: Moraga et al. (2015)

This notebook audits and explores the receptor-binding, trafficking, and early pSTAT6 data selected for Phase 1 of the QSP project.

**Evidence boundary.** The binding and trafficking tables are numerical values reported by the authors. The pSTAT6 time courses are approximate bar heights digitized from published raster figures because point-level source data are not present in the repository. They are useful for exploratory shape checks and weak calibration constraints, but they are not raw measurements.

## Questions

1. Are units, condition labels, and parameter identities internally consistent?
2. Which ligand variants and experimental perturbations are covered?
3. How do affinity, residence time, and pSTAT6 trajectory features co-vary?
4. How strongly does endocytosis inhibition alter the pSTAT6 time course?
5. Which data gaps must be closed before likelihood-based calibration?

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 4)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing pyproject.toml")

ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed" / "phase1_receptor"
META_DIR = ROOT / "data" / "metadata"
ROOT

In [ ]:
with (META_DIR / "moraga2015_source_manifest.json").open(encoding="utf-8") as handle:
    manifest = json.load(handle)

kinetics = pd.read_csv(DATA_DIR / "moraga2015_binding_kinetics.csv")
trafficking = pd.read_csv(DATA_DIR / "moraga2015_trafficking_parameters.csv")
observations = pd.read_csv(DATA_DIR / "moraga2015_pstat6_digitized.csv")

citation = manifest["citation"]
display(Markdown(
    f"**Source:** {citation['authors_short']} ({citation['year']}), "
    f"[{citation['title']}](https://doi.org/{citation['doi']})."
))
print(f"{len(kinetics)} kinetic parameter rows, {len(trafficking)} trafficking rows, "
      f"{len(observations)} digitized observations")

## 1. Data contract and QC

QC fails loudly for schema errors, duplicated record IDs, non-positive rate constants, or an inconsistent derived \(K_D = k_\mathrm{off}/k_\mathrm{on}\). Plausibility bounds are flags, not deletion rules.

In [ ]:
required = {
    "kinetics": {
        "record_id", "variant", "parameter_set", "kon_M_inv_s", "koff_s_inv",
        "kd_nM", "source_locator", "evidence_class"
    },
    "trafficking": {"parameter", "value", "unit", "source_locator", "evidence_class"},
    "observations": {
        "record_id", "experiment_id", "figure_panel", "cell_line", "observable",
        "variant", "dose_nM", "time_min", "condition", "value", "value_unit",
        "extraction_method", "evidence_class"
    },
}
frames = {"kinetics": kinetics, "trafficking": trafficking, "observations": observations}
for name, expected in required.items():
    missing = expected - set(frames[name].columns)
    assert not missing, f"{name}: missing columns {sorted(missing)}"
    assert frames[name]["record_id"].is_unique if "record_id" in frames[name] else True

assert (kinetics[["kon_M_inv_s", "koff_s_inv", "kd_nM"]] > 0).all().all()
kd_recomputed = kinetics["koff_s_inv"] / kinetics["kon_M_inv_s"] * 1e9
np.testing.assert_allclose(kinetics["kd_nM"], kd_recomputed, rtol=1e-5)
assert (observations["time_min"] >= 0).all()
assert observations["value"].between(0, 150).all()

qc = pd.DataFrame({
    "table": frames.keys(),
    "rows": [len(frame) for frame in frames.values()],
    "duplicate_record_ids": [
        int(frame["record_id"].duplicated().sum()) if "record_id" in frame else np.nan
        for frame in frames.values()
    ],
    "missing_cells": [int(frame.isna().sum().sum()) for frame in frames.values()],
})
display(qc)

missingness = observations.isna().mean().sort_values(ascending=False)
display(missingness[missingness > 0].rename("missing_fraction").to_frame())

Missing values in `dose_nM` for Figure 7D are intentional: the main-figure legend does not state the ligand concentration. Missing replicate and dispersion fields reflect unavailable point-level metadata, not zero variance.

In [ ]:
inventory = (
    observations.groupby(
        ["experiment_id", "figure_panel", "cell_line", "condition"], dropna=False
    )
    .agg(
        rows=("record_id", "size"),
        variants=("variant", lambda x: ", ".join(sorted(x.unique()))),
        time_min=("time_min", "min"),
        time_max=("time_min", "max"),
        dose_nM=("dose_nM", lambda x: ", ".join(map(str, sorted(x.dropna().unique()))) or "not stated"),
        evidence_class=("evidence_class", "first"),
    )
    .reset_index()
)
display(inventory)

## 2. Binding-kinetic landscape

Table 3 contains author-reported association and dissociation rates. Table 2 is a simulation parameter set. For C4, D7, and C10, the paper states that no rate constants were available: \(k_\mathrm{off}\) was calculated from measured \(K_D\), while a common \(k_\mathrm{on}\) was imposed.

In [ ]:
kinetics = kinetics.assign(
    residence_half_life_min=np.log(2) / kinetics["koff_s_inv"] / 60,
    log10_kon=np.log10(kinetics["kon_M_inv_s"]),
    log10_koff=np.log10(kinetics["koff_s_inv"]),
)

display(
    kinetics[[
        "variant", "parameter_set", "kon_M_inv_s", "koff_s_inv", "kd_nM",
        "residence_half_life_min", "evidence_class"
    ]].sort_values(["parameter_set", "kd_nM"])
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(
    data=kinetics, x="kon_M_inv_s", y="koff_s_inv", hue="evidence_class",
    style="parameter_set", s=110, ax=axes[0]
)
for row in kinetics.itertuples():
    axes[0].annotate(row.variant, (row.kon_M_inv_s, row.koff_s_inv), xytext=(5, 4),
                     textcoords="offset points", fontsize=9)
axes[0].set(xscale="log", yscale="log", xlabel=r"$k_{on}$ (M$^{-1}$ s$^{-1}$)",
            ylabel=r"$k_{off}$ (s$^{-1}$)", title="Association versus dissociation")

sns.scatterplot(
    data=kinetics, x="kd_nM", y="residence_half_life_min", hue="evidence_class",
    style="parameter_set", s=110, legend=False, ax=axes[1]
)
for row in kinetics.itertuples():
    axes[1].annotate(row.variant, (row.kd_nM, row.residence_half_life_min),
                     xytext=(5, 4), textcoords="offset points", fontsize=9)
axes[1].set(xscale="log", yscale="log", xlabel=r"derived $K_D$ (nM)",
            ylabel="residence half-life (min)", title="Affinity and residence time")
plt.tight_layout()

## 3. Trafficking priors

These rates are model estimates from Table 1. Their half-lives expose the implied separation of endocytosis, recycling, and degradation timescales.

In [ ]:
rate_priors = trafficking.query("unit == 's^-1'").copy()
rate_priors["half_life_min"] = np.log(2) / rate_priors["value"] / 60
rate_priors["half_life_h"] = rate_priors["half_life_min"] / 60
display(rate_priors)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=rate_priors, x="parameter", y="half_life_min", color="#4C78A8", ax=ax)
ax.set(yscale="log", xlabel="", ylabel="implied half-life (min)",
       title="Timescales implied by Moraga Table 1")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)
plt.tight_layout()

## 4. Digitized pSTAT6 time courses

The line segments below connect digitized bar heights. They do not imply continuous observation, and the omitted error bars must not be interpreted as zero uncertainty.

In [ ]:
fig5 = observations.query("experiment_id == 'moraga2015_fig5c'").copy()
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=fig5, x="time_min", y="value", hue="variant", marker="o", ax=ax)
ax.set(xscale="log", xlabel="time (min)", ylabel="% pSTAT6",
       title="A549, 200 nM agonist (digitized Figure 5C)")
ax.legend(title="IL-13 variant", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

In [ ]:
def summarize_curve(frame: pd.DataFrame) -> pd.Series:
    frame = frame.sort_values("time_min")
    t = frame["time_min"].to_numpy(float)
    y = frame["value"].to_numpy(float)
    peak_index = int(np.argmax(y))
    return pd.Series({
        "n_timepoints": len(frame),
        "time_minimum": t.min(),
        "time_maximum": t.max(),
        "peak_pstat6": y[peak_index],
        "time_to_peak_min": t[peak_index],
        "auc_pct_min": np.trapz(y, t),
        "late_to_peak_ratio": y[-1] / y[peak_index] if y[peak_index] else np.nan,
    })

feature_rows = []
feature_keys = ["experiment_id", "figure_panel", "cell_line", "variant", "condition"]
for keys, group in observations.groupby(feature_keys, dropna=False):
    row = dict(zip(feature_keys, keys))
    row.update(summarize_curve(group).to_dict())
    feature_rows.append(row)
features = pd.DataFrame(feature_rows)

display(features.sort_values(["experiment_id", "variant", "condition"]))

fig5_features = features.query("experiment_id == 'moraga2015_fig5c'").sort_values("auc_pct_min")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=fig5_features, x="variant", y="auc_pct_min", color="#F28E2B", ax=axes[0])
axes[0].set(xlabel="IL-13 variant", ylabel="AUC (% pSTAT6 · min)", title="Integrated response")
sns.barplot(data=fig5_features, x="variant", y="late_to_peak_ratio", color="#59A14F", ax=axes[1])
axes[1].set(xlabel="IL-13 variant", ylabel="final / peak", title="Response persistence")
plt.tight_layout()

## 5. Exploratory binding-to-signaling link

The comparison below uses the Table 2 model scenario because it covers exactly the Figure 5C variants. Therefore the result is an internal consistency check of the published modeling setup, not validation against five independently measured rate pairs.

In [ ]:
scenario = kinetics.query("parameter_set == 'table2_model_scenario'")
link = fig5_features.merge(
    scenario[["variant", "kd_nM", "koff_s_inv", "residence_half_life_min"]], on="variant", how="inner"
)
rho_auc, p_auc = spearmanr(link["kd_nM"], link["auc_pct_min"])
rho_persist, p_persist = spearmanr(link["kd_nM"], link["late_to_peak_ratio"])
print(f"Spearman KD vs AUC: rho={rho_auc:.3f}, p={p_auc:.3f}, n={len(link)}")
print(f"Spearman KD vs persistence: rho={rho_persist:.3f}, p={p_persist:.3f}, n={len(link)}")

display(link[["variant", "kd_nM", "koff_s_inv", "auc_pct_min", "late_to_peak_ratio"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=link, x="kd_nM", y="auc_pct_min", s=110, ax=axes[0])
sns.scatterplot(data=link, x="kd_nM", y="late_to_peak_ratio", s=110, ax=axes[1])
for ax, ycol in zip(axes, ["auc_pct_min", "late_to_peak_ratio"]):
    for row in link.itertuples():
        ax.annotate(row.variant, (row.kd_nM, getattr(row, ycol)), xytext=(5, 4),
                    textcoords="offset points")
    ax.set_xscale("log")
axes[0].set(xlabel=r"Table 2 $K_D$ (nM)", ylabel="AUC (% pSTAT6 · min)",
            title="Affinity versus integrated signal")
axes[1].set(xlabel=r"Table 2 $K_D$ (nM)", ylabel="final / peak",
            title="Affinity versus persistence")
plt.tight_layout()

warnings.warn("n=5 and partially assumed kinetics: do not interpret p-values as confirmatory evidence.")

## 6. Endocytosis perturbation

Figure 7D compares pSTAT6 with and without 100 µM EHT1864. The effect is summarized both pointwise and by trajectory AUC. Because the digitized series do not include uncertainty, no significance test is performed.

In [ ]:
fig7 = observations.query("experiment_id == 'moraga2015_fig7d'").copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, variant in zip(axes, ["WT", "D7", "A11"]):
    subset = fig7.query("variant == @variant")
    sns.lineplot(data=subset, x="time_min", y="value", hue="condition", marker="o", ax=ax)
    ax.set(title=variant, xlabel="time (min)", ylabel="% pSTAT6")
    if ax is not axes[-1]:
        ax.get_legend().remove()
axes[-1].legend(title="condition", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("HeLa pSTAT6 with endocytosis inhibition (digitized Figure 7D)", y=1.03)
plt.tight_layout()

In [ ]:
control_auc = (
    features.query("experiment_id == 'moraga2015_fig7d'")
    .pivot(index="variant", columns="condition", values="auc_pct_min")
    .rename_axis(columns=None)
)
control_auc["retained_auc_fraction"] = (
    control_auc["endocytosis_inhibited"] / control_auc["control"]
)
control_auc["auc_reduction_percent"] = 100 * (1 - control_auc["retained_auc_fraction"])
display(control_auc.sort_values("auc_reduction_percent", ascending=False))

pointwise = fig7.pivot_table(
    index=["variant", "time_min"], columns="condition", values="value"
).reset_index()
pointwise["retained_signal_fraction"] = (
    pointwise["endocytosis_inhibited"] / pointwise["control"].replace(0, np.nan)
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=pointwise, x="time_min", y="retained_signal_fraction", hue="variant", marker="o", ax=ax)
ax.axhline(1, color="black", lw=1, ls="--")
ax.set(xlabel="time (min)", ylabel="EHT1864 / control",
       title="Pointwise signal retained under endocytosis inhibition")
plt.tight_layout()

## 7. Coverage and decision log

In [ ]:
coverage = (
    observations.groupby(["experiment_id", "variant", "condition"])
    .agg(n_times=("time_min", "nunique"), min_time=("time_min", "min"), max_time=("time_min", "max"))
    .reset_index()
    .pivot_table(index="variant", columns=["experiment_id", "condition"], values="n_times", fill_value=0)
)
display(coverage)

limitations = pd.DataFrame({"known_limitation": manifest["known_limitations"]})
display(limitations)

## Interpretation for Phase 1

**Supported now**

- Explore how ligand residence time and trafficking timescales shape early pSTAT6 amplitude and persistence.
- Use the author-reported rates as informative priors or fixed scenario parameters with evidence labels preserved.
- Use the digitized pSTAT6 curves for qualitative trajectory checks and weakly weighted calibration targets.

**Not supported yet**

- A measurement-error likelihood: point-level replicates and a defined dispersion statistic are absent.
- Identifying \(k_\mathrm{on}\) and \(k_\mathrm{off}\) separately for C4, D7, and C10 from Table 2.
- Assigning a Figure 7D ligand concentration without an additional source.
- Confirmatory binding-to-signaling inference from five partly modeled kinetic pairs.

**Next data action:** request author-supplied pSTAT6 tables or FCS files. If unavailable, repeat digitization from a vector or higher-resolution source and capture upper and lower error-bar coordinates with an explicit observation-error model.